# Age structure → ITL2 (counts, bands, median age), 1991–2025

Builds **age structure per ITL2 region per year** for comparing regions' demography. Two outputs:

- `clean/age_single_year_itl2.csv` — the full distribution: persons by **single year of age** per ITL2 × year (long), with the open-ended top group flagged.
- `clean/age_itl2.csv` — summary per ITL2 × year: total population, **age-band counts**, band shares, **median age**, and child / old-age **dependency ratios**.
- `clean/median_age_itl2.csv` — just **median age** per ITL2 × year (1 d.p.), for quick use.

**Counts first, on purpose.** LA single-year counts are SUMMED to ITL2 (exact), and every summary — including median age — is computed *from the ITL2 counts*. We never average LA medians or shares.

**Source:** Nomis **NM_2002_1** (ONS/NRS/NISRA mid-year population estimates, local authority based, by single year of age), persons (`gender=0`), all 361 UK LAs (`geography` list from the Nomis query builder — the same one `build/population.py` uses). This is the **same release as `clean/population_itl2.csv`**, so totals match it exactly (checked in §4).

**What Nomis publishes, and how it's handled:**

| | ages published | top (open-ended) group |
|---|---|---|
| England & Wales, 1991–2000 | 0 … 84 | **85+** (ages 85–89 / 90+ not published) |
| England & Wales, 2001–2025 | 0 … 89 | 90+ |
| Scotland, 1991–2025 | 0 … 89 | 90+ |
| Northern Ireland | from **2001** only | 90+ |

So the single-year file stores `age` as the lower bound of each group with `open_ended = True` on the top group (85 or 90). **Median age and all bands up to 84 are unaffected**; the `85_plus` band is complete in every year. Every LA-year is checked to sum exactly to Nomis's *All Ages* figure. TLN0 (Northern Ireland) is absent before 2001.

**Query trimmed from the query-builder URL:** the original also requested 5-year age bands; we keep only the single years (`c_age=101…191`), 85+ (`210`, needed for E&W 1991–2000) and All Ages (`200`, as a check), and `select=` only the four columns we use. Paged past Nomis's 25,000-row cap (~1.2m rows, a few minutes).

**Median age** uses the standard grouped-data interpolation, treating each year of age *a* as [a, a+1): median = a + (N/2 − people younger than a) / (people aged a). The top group never contains the median.

**Split LA:** North Ayrshire is assigned wholly to TLM9 by `build/geography.py`; this notebook stops if the spine ever lists an LA twice.

In [1]:
import io, time, urllib.request
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
SPINE      = BASE / "clean" / "geography_spine.csv"
CROSSWALK  = BASE / "lib" / "la_code_crosswalk.csv"
POP_ITL2   = BASE / "clean" / "population_itl2.csv"         # Nomis all-ages totals, for the cross-check
RAW_DIR    = BASE / "raw" / "population"
OUT_SINGLE = BASE / "clean" / "age_single_year_itl2.csv"
OUT        = BASE / "clean" / "age_itl2.csv"
OUT_MEDIAN = BASE / "clean" / "median_age_itl2.csv"

GEOGRAPHY = ("1778384897...1778384901,1778384941,1778384950,1778385143...1778385146,1778385159,1778384902...1778384905,1778384942,1778384943,1778384956,1778384957,1778385033...1778385044,1778385124...1778385138,1778384906...1778384910,1778384958,1778385139...1778385142,1778385154...1778385158,1778384911...1778384914,1778384954,1778384955,1778384965...1778384972,1778385045...1778385058,1778385066...1778385072,1778384915...1778384917,1778384944,1778385078...1778385085,1778385100...1778385104,1778385112...1778385117,1778385147...1778385153,1778384925...1778384928,1778384948,1778384949,1778384960...1778384964,1778384986...1778384997,1778385015...1778385020,1778385059...1778385065,1778385086...1778385088,1778385118...1778385123,1778385160...1778385192,1778384929...1778384940,1778384953,1778384981...1778384985,1778385004...1778385014,1778385021...1778385032,1778385073...1778385077,1778385089...1778385099,1778385105...1778385111,1778384918...1778384924,1778384945...1778384947,1778384951,1778384952,1778384973...1778384980,1778384998...1778385003,1778384959,1778385193...1778385257")
AGE_ALL, AGE_85PLUS = 200, 210                              # Nomis c_age codes: All Ages, Aged 85+
SINGLE = list(range(101, 192))                              # 101 = Age 0 ... 190 = Age 89, 191 = Aged 90+
URL = ("https://www.nomisweb.co.uk/api/v01/dataset/NM_2002_1.data.csv"
       f"?geography={GEOGRAPHY}&date=first-latest&gender=0"
       f"&c_age={AGE_ALL},{AGE_85PLUS},101...191&measures=20100"
       "&select=date,geography_code,c_age,obs_value")

spine = pd.read_csv(SPINE)
assert spine.la_code.is_unique, "spine lists an LA twice - rebuild it with build/geography.py"
lad2itl   = dict(zip(spine.la_code, spine.itl2_code))
itl2_name = dict(zip(spine.itl2_code, spine.itl2_name))
remap = dict(zip(*[pd.read_csv(CROSSWALK, dtype=str)[c] for c in ("old_code","new_code")])) if CROSSWALK.exists() else {}
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Fetch from Nomis (paged), save raw

25,000 rows per request until a short page; the extract must be a complete, duplicate-free grid. Saved, dated, to `raw/population/`.

In [2]:
def read_page(url, tries=3):
    for k in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=300) as r:
                return pd.read_csv(io.BytesIO(r.read()), encoding="utf-8-sig")
        except Exception as e:
            if k == tries - 1: raise
            print(f"  retry ({e})"); time.sleep(5)

def fetch_nomis_all(url, limit=25000):
    frames, offset = [], 0
    while True:
        part = read_page(f"{url}&recordlimit={limit}&recordoffset={offset}")
        frames.append(part)
        if len(part) < limit:
            break
        offset += limit
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip().upper() for c in df.columns]
    return df

raw = fetch_nomis_all(URL)
keys = ["DATE","GEOGRAPHY_CODE","C_AGE"]
assert not raw.duplicated(keys).any(), "duplicate rows - paging overlapped"
assert len(raw) == raw.DATE.nunique() * raw.GEOGRAPHY_CODE.nunique() * raw.C_AGE.nunique(), "incomplete grid - paging lost rows"
raw_path = RAW_DIR / f"nomis_population_age_{date.today().isoformat()}.csv"
raw.to_csv(raw_path, index=False)
print(f"Saved raw -> {raw_path.name}: {len(raw):,} rows | {raw.GEOGRAPHY_CODE.nunique()} LAs | "
      f"{raw.DATE.min()}-{raw.DATE.max()} | {raw.C_AGE.nunique()} age codes")

Saved raw -> nomis_population_age_2026-10-07.csv: 1,175,055 rows | 361 LAs | 1991-2025 | 93 age codes


## 2. Tidy to LA × year × age group, with the right top group

Per LA-year: if ages 85–89 and 90+ are published, use single years 0–89 plus 90+; otherwise (E&W 1991–2000) use single years 0–84 plus Nomis's 85+. Each LA-year must then sum exactly to All Ages. LA-years with no data at all (NI before 2001) are dropped.

In [3]:
raw["la_code"] = raw.GEOGRAPHY_CODE.astype(str).str.strip().replace(remap)
assert not raw.duplicated(["DATE","la_code","C_AGE"]).any(), "an LA appears twice after crosswalk"
w = raw.pivot_table(index=["la_code","DATE"], columns="C_AGE", values="OBS_VALUE")   # one row per LA-year

full = w[list(range(186, 192))].notna().all(axis=1)          # ages 85-89 and 90+ published
top85 = ~full & w[AGE_85PLUS].notna()
print(f"LA-years with full detail (top 90+): {full.sum():,} | top-coded at 85+: {top85.sum():,} | "
      f"no data: {(~full & ~top85).sum():,}")

def groups(row_mask, ages_codes, top_code, top_age):
    g = w.loc[row_mask, ages_codes + [top_code]].copy()
    g.columns = [c - 101 for c in ages_codes] + [top_age]       # c_age 101 = age 0
    return g

parts = [groups(full, list(range(101, 191)), 191, 90),           # 0-89 + 90+
         groups(top85, list(range(101, 186)), AGE_85PLUS, 85)]    # 0-84 + 85+
la = pd.concat([p.rename_axis(columns="age").stack(future_stack=True).rename("persons").reset_index()
                for p in parts], ignore_index=True)
la["open_ended"] = la.groupby(["la_code","DATE"]).age.transform("max").eq(la.age)
la = la.rename(columns={"DATE":"year"})

# every LA-year sums exactly to All Ages, with no gaps
assert la.persons.notna().all(), "missing single-year values inside a published LA-year"
chk = la.groupby(["la_code","year"]).persons.sum().to_frame("sum").join(
      w[AGE_ALL].rename_axis(["la_code","year"]).rename("all_ages"))
assert (chk["sum"] == chk["all_ages"]).all(), chk[chk["sum"] != chk["all_ages"]].head()
print("Every LA-year sums exactly to Nomis All Ages.")

la["itl2_code"] = la.la_code.map(lad2itl)
print("LA codes not in spine (dropped):", sorted(la.loc[la.itl2_code.isna(), "la_code"].unique()) or "none")
print("Spine LAs not in Nomis:", sorted(set(spine.la_code) - set(la.la_code)) or "none")
la = la.dropna(subset=["itl2_code"])
print(f"{la.la_code.nunique()} LAs | {la.year.min()}-{la.year.max()} | "
      f"NI from {la[la.la_code.str.startswith('N')].year.min()}")

LA-years with full detail (top 90+): 9,345 | top-coded at 85+: 3,180 | no data: 0


Every LA-year sums exactly to Nomis All Ages.
LA codes not in spine (dropped): none
Spine LAs not in Nomis: none


361 LAs | 1991-2025 | NI from 2001


## 3. Sum to ITL2 → single-year distribution

Exact aggregation. ITL2 regions don't cross country borders, so each ITL2-year has one consistent top group (asserted).

In [4]:
single = (la.groupby(["itl2_code","year","age"], as_index=False)
            .agg(persons=("persons","sum"), open_ended=("open_ended","first")))
assert la.groupby(["itl2_code","year","age"]).open_ended.nunique().eq(1).all(), "mixed top groups within an ITL2-year"
single["itl2_name"] = single.itl2_code.map(itl2_name)
single = single[["itl2_code","itl2_name","year","age","open_ended","persons"]].sort_values(["year","itl2_code","age"]).reset_index(drop=True)

OUT_SINGLE.parent.mkdir(parents=True, exist_ok=True)
single.to_csv(OUT_SINGLE, index=False)
print(f"Saved -> {OUT_SINGLE}  ({len(single):,} rows; open_ended marks the top group: 85+ or 90+)")
single.head()

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/age_single_year_itl2.csv  (143,650 rows; open_ended marks the top group: 85+ or 90+)


,itl2_code,itl2_name,year,age,open_ended,persons
0,TLC3,Tees Valley,1991,0,False,9371.0
1,TLC3,Tees Valley,1991,1,False,9234.0
2,TLC3,Tees Valley,1991,2,False,9309.0
3,TLC3,Tees Valley,1991,3,False,9727.0
4,TLC3,Tees Valley,1991,4,False,9580.0


## 4. Summary per ITL2 × year: bands, shares, median age, dependency ratios

Bands (counts, which sum to the total): **0–15, 16–24, 25–49, 50–64, 65–84, 85+**, plus **16–64** (working age) and **65+**. Ratios from the ITL2 counts: child dependency = 0–15 / 16–64; old-age dependency = 65+ / 16–64 (per 100 working-age). `top_age_group` records whether the top group was 85+ or 90+.

In [5]:
BANDS = {"0_15": (0, 15), "16_24": (16, 24), "25_49": (25, 49), "50_64": (50, 64), "65_84": (65, 84), "85_plus": (85, 90)}

def median_age(g):
    """Interpolated median from single-year counts; age a covers [a, a+1)."""
    g = g.sort_values("age")
    n = g.persons.to_numpy(float); cum = np.cumsum(n); half = cum[-1] / 2
    i = np.searchsorted(cum, half)
    assert not g.open_ended.iloc[i], "median falls in the open-ended top group"
    return g.age.iloc[i] + (half - (cum[i] - n[i])) / n[i]

grp = single.groupby(["itl2_code","year"])
summ = grp.persons.sum().rename("population").to_frame()
for name, (lo, hi) in BANDS.items():
    summ[f"pop_{name}"] = single[single.age.between(lo, hi)].groupby(["itl2_code","year"]).persons.sum()
summ["pop_16_64"]   = summ[["pop_16_24","pop_25_49","pop_50_64"]].sum(axis=1)
summ["pop_65_plus"] = summ[["pop_65_84","pop_85_plus"]].sum(axis=1)
assert (summ[[f"pop_{b}" for b in BANDS]].sum(axis=1) == summ.population).all(), "bands don't sum to total"

for b in list(BANDS) + ["16_64","65_plus"]:
    summ[f"share_{b}"] = 100 * summ[f"pop_{b}"] / summ.population
summ["median_age"]      = grp.apply(median_age, include_groups=False)
summ["child_dep_ratio"] = 100 * summ.pop_0_15 / summ.pop_16_64
summ["old_dep_ratio"]   = 100 * summ.pop_65_plus / summ.pop_16_64
summ["top_age_group"]   = single[single.open_ended].set_index(["itl2_code","year"]).age.map({85: "85+", 90: "90+"})

age = summ.reset_index()
age.insert(1, "itl2_name", age.itl2_code.map(itl2_name))
age = age.sort_values(["year","itl2_code"]).reset_index(drop=True)
age.to_csv(OUT, index=False, float_format="%.4f")
print(f"Saved -> {OUT}  ({len(age)} rows)")

age[["itl2_code","itl2_name","year","median_age"]].round({"median_age": 1}).to_csv(OUT_MEDIAN, index=False)
print(f"Saved -> {OUT_MEDIAN}")
print("Columns:", list(age.columns))
age.head()

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/age_itl2.csv  (1600 rows)
Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/median_age_itl2.csv
Columns: ['itl2_code', 'itl2_name', 'year', 'population', 'pop_0_15', 'pop_16_24', 'pop_25_49', 'pop_50_64', 'pop_65_84', 'pop_85_plus', 'pop_16_64', 'pop_65_plus', 'share_0_15', 'share_16_24', 'share_25_49', 'share_50_64', 'share_65_84', 'share_85_plus', 'share_16_64', 'share_65_plus', 'median_age', 'child_dep_ratio', 'old_dep_ratio', 'top_age_group']


,itl2_code,itl2_name,year,population,pop_0_15,pop_16_24,pop_25_49,pop_50_64,pop_65_84,pop_85_plus,...,share_25_49,share_50_64,share_65_84,share_85_plus,share_16_64,share_65_plus,median_age,child_dep_ratio,old_dep_ratio,top_age_group
0,TLC3,Tees Valley,1991,656181.0,146084.0,81905.0,229730.0,104530.0,86207.0,7725.0,...,35.010157,15.930056,13.137686,1.177267,63.422287,14.314953,34.857909,35.102423,22.570855,85+
1,TLC4,"Northumberland, Durham and Tyne & Wear",1991,1930805.0,385964.0,248019.0,669677.0,315599.0,284776.0,26770.0,...,34.683824,16.345462,14.749081,1.386468,63.874653,16.135550,36.546811,31.295351,25.261272,85+
2,TLD1,Cumbria,1991,486263.0,93812.0,58258.0,167760.0,82242.0,76316.0,7875.0,...,34.499849,16.913070,15.694388,1.619494,63.393678,17.313882,38.524220,30.432752,27.311685,85+
3,TLD3,Greater Manchester,1991,2553564.0,545339.0,345381.0,892318.0,383282.0,352421.0,34823.0,...,34.944023,15.009688,13.801142,1.363702,63.479161,15.164844,34.580145,33.642529,23.889484,85+
4,TLD4,Lancashire,1991,1402421.0,290211.0,175314.0,477428.0,222417.0,213627.0,23424.0,...,34.043130,15.859503,15.232730,1.670255,62.403444,16.902984,37.039690,33.160946,27.086621,85+


## 5. Checks

1. Coverage by year (46 ITL2 from 2001; 45 before, no NI).
2. Totals vs `clean/population_itl2.csv` — same Nomis dataset, so the gap should be **0** in every region-year.
3. UK totals and UK median age (recomputed from UK-summed counts; UK only from 2001, when NI starts).
4. Latest-year ranking — oldest and youngest ITL2s by median age.

In [6]:
cov = age.groupby("year").itl2_code.nunique()
print("ITL2 regions by year:", cov.groupby(cov).apply(lambda s: f"{s.index.min()}-{s.index.max()}").to_dict())

if POP_ITL2.exists():
    nomis = pd.read_csv(POP_ITL2)
    cmp = age.merge(nomis, on=["itl2_code","year"], suffixes=("","_nomis"), how="left")
    gap = (cmp.population - cmp.population_nomis).abs()
    print(f"\nvs clean/population_itl2.csv: {len(cmp)} region-years matched, max |gap| = {gap.max():.0f} persons")

uk = single[single.year >= 2001].groupby(["year","age"], as_index=False).agg(persons=("persons","sum"), open_ended=("open_ended","first"))
uk_tab = pd.DataFrame({"population_m": uk.groupby("year").persons.sum() / 1e6,
                       "median_age":  uk.groupby("year").apply(median_age, include_groups=False)})
print("\nUK totals and median age (2001+):"); print(uk_tab.round(2).T.to_string())

latest = age[age.year == age.year.max()]
cols = ["itl2_name","median_age","share_65_plus","old_dep_ratio"]
print(f"\nOldest ITL2s by median age, {age.year.max()}:");   print(latest.nlargest(5, "median_age")[cols].round(1).to_string(index=False))
print(f"\nYoungest ITL2s by median age, {age.year.max()}:"); print(latest.nsmallest(5, "median_age")[cols].round(1).to_string(index=False))

first = age[age.year == age.year.min()].set_index("itl2_code").median_age
chg = (latest.set_index("itl2_code").median_age - first).dropna()
print(f"\nMedian-age change {age.year.min()}-{age.year.max()}: smallest {chg.min():+.1f} ({itl2_name[chg.idxmin()]}), "
      f"largest {chg.max():+.1f} ({itl2_name[chg.idxmax()]})")

ITL2 regions by year: {45: '1991-2000', 46: '2001-2025'}

vs clean/population_itl2.csv: 1600 region-years matched, max |gap| = 0 persons

UK totals and median age (2001+):
year           2001   2002   2003   2004   2005   2006   2007   2008   2009   2010   2011   2012   2013   2014   2015   2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
population_m  59.11  59.37  59.64  59.95  60.41  60.83  61.32  61.82  62.26  62.76  63.29  63.71  64.14  64.62  65.09  65.61  65.96  66.29  66.63  66.74  66.98  67.68  68.59  69.26  69.48
median_age    37.87  38.13  38.36  38.56  38.72  38.89  39.01  39.15  39.32  39.47  39.59  39.77  39.90  40.02  40.07  40.09  40.17  40.24  40.38  40.57  40.77  40.78  40.68  40.70  40.91

Oldest ITL2s by median age, 2025:
                          itl2_name  median_age  share_65_plus  old_dep_ratio
              Highlands and Islands        49.0           25.7           43.3
       Cornwall and Isles of Scilly        48.3           26.5           